# 04.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/06-feature-engineering/solutions.ipynb)

## Exercise 1

R = 1 day (31 Aug → 1 Sep), F = 3, M = £60, average basket = £20, log(1 + 60) = ln 61 ≈ 4.111.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

d = pd.DataFrame({"date": pd.to_datetime(["2011-08-31", "2011-08-01", "2011-07-01"]), "amount": [12, 18, 30]})
cutoff = pd.Timestamp("2011-09-01")
R = (cutoff - d["date"].max()).days
F, M = len(d), d["amount"].sum()
print(f"R={R}, F={F}, M={M}, avg basket={M / F:.0f}, log1p(M)={np.log1p(M):.3f}")

R=1, F=3, M=60, avg basket=20, log1p(M)=4.111


## Exercise 2

In [2]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")                       # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])
features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

3,317 customers, churn rate 43.6%


In [3]:
cust["spend_q"] = pd.qcut(cust["spend"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
cust.groupby("spend_q", observed=True)["churned"].agg(["count", "mean"]).round(3)

,count,mean
spend_q,,
Q1 lowest,830,0.643
Q2,829,0.551
Q3,829,0.382
Q4 highest,829,0.166


Also clean and monotonic: 64% churn in the lowest-spending quarter, falling to 17% in the highest. Spend and recency both carry signal, which is why the model uses several features.

## Exercise 3

In [4]:
cust["products_per_order"] = cust["products"] / cust["orders"]
cust["ppo_q"] = pd.qcut(cust["products_per_order"], 4)
cust.groupby("ppo_q", observed=True)["churned"].agg(["count", "mean"]).round(3)

,count,mean
ppo_q,,
"(0.249, 7.667]",834,0.478
"(7.667, 13.667]",829,0.427
"(13.667, 23.0]",843,0.414
"(23.0, 262.0]",811,0.424


A high value means wide baskets: many different products per order, typical of shops and resellers restocking. But the signal is weak: churn is 48% in the lowest quarter and about 42% in the other three, compared with 64% → 17% across spend quartiles. Not every sensible-sounding feature carries information. Checking a feature's churn rate by quartile like this takes one line and saves you from adding noise.

## Exercise 4

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

sept = sales[(sales["InvoiceDate"] >= "2011-09-01") & (sales["InvoiceDate"] < "2011-10-01")]
cust["orders_in_september"] = sept.groupby("Customer ID")["Invoice"].nunique().reindex(cust.index).fillna(0)
cols = features + ["orders_in_september"]
tr, te = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(tr[cols], tr["churned"])
print(f"AUC with the leaky feature: {roc_auc_score(te['churned'], m.predict_proba(te[cols])[:, 1]):.3f}")

AUC with the leaky feature: 0.853


The AUC jumps from 0.741 to 0.853 because September is *inside the target window*: anyone who ordered in September did not churn, by definition. On 1 September, when ShopWave needs the prediction, this column does not exist yet. That is **leakage** (lesson 04.2): a feature that would not be available at prediction time.